# 🚀 High-Precision Semiconductor Image Restoration (NAFNet-SR)

This notebook continues the accepted epoch-8 **Research NAFNet-SR** checkpoint (`28.73 dB` no TTA; `28.84 dB` with TTA), including its trained **2D local/FFT dual-domain mixer** and **beta-NLL uncertainty head**.

> **This is the NAFNet-only workflow and does not instantiate MambaIRv2.** Use [train_fusion_colab.ipynb](https://colab.research.google.com/github/kmbeddedd/semicon_2026/blob/Kunal/train_fusion_colab.ipynb) to train the full MambaIRv2 Base fusion model.

### **GPU Optimizations & Metrology Features Enabled**
- **Global Bicubic Residual Learning**: Direct high-frequency residual optimization
- **Model Exponential Moving Average (EMA)**: Stable validation and checkpoint selection
- **Ortho-Normalized Frequency & MS-SSIM Losses**: Balanced spatial and spectral gradients
- **Dual-Domain Bottleneck Mixer**: Gated local convolution and 2D FFT features
- **Calibrated Uncertainty Head**: Heteroscedastic beta-NLL auxiliary supervision
- **8-Fold Test-Time Augmentation (TTA)**: Dihedral ensemble for maximum evaluation accuracy
- **Tensor Core Acceleration**: PyTorch Automatic Mixed Precision (`AMP FP16` with `GradScaler`)


## Step 1: Verify GPU & Tensor Cores


In [ ]:
!nvidia-smi


## Step 2: Clone Repository (Branch `Kunal`)


In [ ]:
!git clone -b Kunal https://github.com/kmbeddedd/semicon_2026.git
%cd semicon_2026


## Step 3: Install Dependencies


In [ ]:
!pip install -r requirements.txt


## Step 4: Extract Dataset & Create Validation Split


In [ ]:
import os, zipfile, glob, shutil, random
from google.colab import drive

# Mount Google Drive for permanent dataset storage
drive.mount('/content/drive')

# Check for dataset in Drive or local directory
drive_train_zip = '/content/drive/MyDrive/train.zip'
local_train_zip = 'train.zip'
zip_path = drive_train_zip if os.path.exists(drive_train_zip) else (local_train_zip if os.path.exists(local_train_zip) else None)

if zip_path:
    print(f'Extracting dataset from: {zip_path}...')
    os.makedirs('data', exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('data')
    print('Training dataset extraction complete!')
else:
    print('⚠️ train.zip not found! Please upload train.zip to your Google Drive MyDrive root.')

# Check for test set
drive_test_zip = '/content/drive/MyDrive/Test_NoisyLR.zip'
local_test_zip = 'Test_NoisyLR.zip'
test_zip_path = drive_test_zip if os.path.exists(drive_test_zip) else (local_test_zip if os.path.exists(local_test_zip) else None)

if test_zip_path:
    print(f'Extracting test dataset from: {test_zip_path}...')
    os.makedirs('data/test', exist_ok=True)
    with zipfile.ZipFile(test_zip_path, 'r') as z:
        z.extractall('data/test')
    print('Test dataset extraction complete!')

# Create Train / Val split (10% validation)
if os.path.exists('data/train/NoisyLR') and not os.path.exists('data/val'):
    random.seed(42)
    os.makedirs('data/val/NoisyLR', exist_ok=True)
    os.makedirs('data/val/GT', exist_ok=True)
    files = sorted(glob.glob('data/train/NoisyLR/*.npy'))
    val_files = random.sample(files, k=int(len(files) * 0.1))
    for f in val_files:
        fname = os.path.basename(f)
        shutil.move(f, os.path.join('data/val/NoisyLR', fname))
        shutil.move(os.path.join('data/train/GT', fname), os.path.join('data/val/GT', fname))
    print(f'Created Val Split: {len(val_files)} samples moved to data/val/')


## Step 5: Continue Resumable Research Fine-Tuning
The trainer now probes real AMP forward/loss/backward steps and chooses the largest safe full-resolution batch within **88% of GPU VRAM** (typically about batch 24 / 13 GiB reserved on a 15 GiB T4). The remaining headroom protects cuDNN/cuFFT workspaces from late OOMs. Data stay on Colab's local disk with two loader workers; checkpoints are written to Google Drive.

In [ ]:
import os, shutil, subprocess

EPOCHS = 20
EXPERIMENT_DIR = '/content/drive/MyDrive/semicon_vram88_weights'
BASE_WEIGHTS = 'weights/best_model.pt'
BEST_WEIGHTS = os.path.join(EXPERIMENT_DIR, 'best_model.pt')
LATEST_WEIGHTS = os.path.join(EXPERIMENT_DIR, 'latest_model.pt')
os.makedirs(EXPERIMENT_DIR, exist_ok=True)

# Keep the accepted baseline until a research checkpoint actually beats it.
if not os.path.exists(BEST_WEIGHTS):
    shutil.copy2(BASE_WEIGHTS, BEST_WEIGHTS)

command = [
    'python', 'train.py',
    '--epochs', str(EPOCHS),
    '--warmup_epochs', '1',
    '--batch_size', '8',  # CPU/no-autotune fallback
    '--auto_batch_size',
    '--target_vram_fraction', '0.88',
    '--max_batch_size', '64',
    '--lr', '1e-5',
    '--extension_lr_multiplier', '1',
    '--scale', '2',
    '--patch_size', '0',
    '--augmentation', 'd4',
    '--psnr_polish_epochs', '5',
    '--num_workers', '2',
    '--no_cache',
    '--ema_decay', '0.999',
    '--seed', '42',
    '--spectral_mixer',
    '--uncertainty_head',
    '--w_nll', '0.02',
    '--nll_beta', '0.5',
    '--save_dir', EXPERIMENT_DIR,
]
if os.path.exists(LATEST_WEIGHTS):
    command += ['--resume', LATEST_WEIGHTS]
    print(f'Resuming from {LATEST_WEIGHTS}')
else:
    command += ['--init_weights', BASE_WEIGHTS]
    print(f'Initializing research extension from {BASE_WEIGHTS}')

subprocess.run(command, check=True)


## Step 6: Inspect the Persistent Best Checkpoint


In [ ]:
import torch
checkpoint = torch.load(BEST_WEIGHTS, map_location='cpu')
print('Best checkpoint:', BEST_WEIGHTS)
print('Epoch:', checkpoint.get('epoch'))
print('Validation PSNR:', checkpoint.get('val_psnr'))
print('Validation SSIM:', checkpoint.get('val_ssim'))
print('Model config:', checkpoint.get('model_config', 'legacy base architecture'))


## Step 7: Validate the Winner and Run 8-Fold TTA Test Inference


In [ ]:
# Reproduce validation metrics with the selected checkpoint.
!python eval.py --input_dir data/val/NoisyLR --target_dir data/val/GT --output_dir data/val_research --weights "{BEST_WEIGHTS}" --scale 2 --batch_size 8 --no_tta --check_clean_damage

# Maximum-accuracy D4 ensemble for the hidden test set.
TEST_OUTPUT = '/content/drive/MyDrive/semicon_research_test_outputs'
!python eval.py --input_dir data/test/NoisyLR --output_dir "{TEST_OUTPUT}" --weights "{BEST_WEIGHTS}" --scale 2 --batch_size 8
